In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import boto3
import time
from functools import wraps

In [3]:
import logging 
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

In [4]:
def retry(max_attempts=3,delay=2):
    def decorator(func):
        @wraps(func)
        def wrapper(*args,**kwargs):
            for attempt in range(max_attempts):
                try:
                    return func(*args,**kwargs)
                except Exception as e:
                    if attempt == max_attempts - 1:
                        raise
                    backoff = delay * (2**attempt)
                    time.sleep(backoff)
        return wrapper
    return decorator


In [5]:
@retry()
def show_objects(bucket,simulate_timeout=False):
    try:
        logging.info(f"Listing Objects in AWS S3 | Bucket = {bucket}")
        s3 = boto3.client(
            's3',
            aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID"),
            aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY"),
            region_name = 'ap-south-1'
        )
        if simulate_timeout:
            raise ConnectionError("Connection timeout...")
        response = s3.list_objects_v2(Bucket = bucket)
        if 'Contents' not in response:
            return []
        objects =  [object['Key'] for object in response['Contents']]
        logging.info(f"Found {len(objects)} objects: {objects}")
    except Exception as e:
        logging.info(f"Download failed: {e}")
        raise
    

In [6]:
show_objects('raw-bucket-427763921511-ap-south-1-an')

2026-07-30 17:30:47,801 INFO Listing Objects in AWS S3 | Bucket = raw-bucket-427763921511-ap-south-1-an
2026-07-30 17:30:48,211 INFO Found 8 objects: ['dirty/dealer.csv', 'dirty/inventory.csv', 'dirty/product.csv', 'dirty/sales_logs.jsonl', 'dirty/test_inventory.csv', 'processed/clean/dealer.csv', 'processed/dealer_clean.csv', 'processed/reject/dealer.csv']


In [7]:
show_objects('raw-bucket-427763921511-ap-south-1-an',simulate_timeout=True)

2026-07-30 17:30:48,227 INFO Listing Objects in AWS S3 | Bucket = raw-bucket-427763921511-ap-south-1-an
2026-07-30 17:30:48,242 INFO Download failed: Connection timeout...
2026-07-30 17:30:50,246 INFO Listing Objects in AWS S3 | Bucket = raw-bucket-427763921511-ap-south-1-an
2026-07-30 17:30:50,264 INFO Download failed: Connection timeout...
2026-07-30 17:30:54,271 INFO Listing Objects in AWS S3 | Bucket = raw-bucket-427763921511-ap-south-1-an
2026-07-30 17:30:54,291 INFO Download failed: Connection timeout...


ConnectionError: Connection timeout...